# Music Intelligence
## Diagnóstico competitivo de artistas y estimación de popularidad musical

**Objetivo:** construir un flujo reproducible para describir y comparar artistas con suficiente información en el catálogo disponible y estimar la popularidad observada de canciones mediante Machine Learning.

> **Alcance:** la popularidad de Spotify es una métrica de plataforma. Este notebook no predice éxito futuro, ventas, asistencia a conciertos ni rentabilidad de campañas. Las diferencias entre popularidad real y estimada son señales exploratorias para revisión humana.

### Instrucciones
1. El notebook descarga `spotify_data_clean_readable.csv` directamente desde la rama `main` del repositorio público de GitHub. No es necesario guardar el CSV localmente.
2. Ejecutar las celdas en orden.
3. Revisar las tablas y métricas antes de utilizar resultados en la presentación.
4. Los CSV exportados se guardarán en la misma carpeta de trabajo.

## 1. Librerías y configuración

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

RANDOM_STATE = 42
RUTA_DATOS = "https://raw.githubusercontent.com/Luisfegit/Music-Intelligence/main/spotify_data_clean_readable.csv"
UMBRAL_CANCIONES = 5

## 2. Carga y validación inicial del dataset

Se utiliza el CSV de la primera entrega para mantener continuidad con el análisis exploratorio. Se verifica que estén presentes las columnas necesarias.

In [2]:
df_original = pd.read_csv(RUTA_DATOS)

print(f"Dimensiones originales: {df_original.shape[0]:,} filas y {df_original.shape[1]} columnas")
display(df_original.head())

columnas_requeridas = [
    "track_id", "track_name", "track_popularity", "artist_name",
    "artist_popularity", "artist_followers", "artist_genres",
    "track_number", "track_duration_min", "album_total_tracks",
    "album_release_date", "explicit", "album_type"
]

columnas_faltantes = [col for col in columnas_requeridas if col not in df_original.columns]
if columnas_faltantes:
    raise ValueError(f"Faltan columnas requeridas en el CSV: {columnas_faltantes}")

display(df_original[columnas_requeridas].isna().sum().to_frame("valores_faltantes"))

Dimensiones originales: 8,582 filas y 15 columnas


,track_id,track_name,track_number,track_popularity,explicit,artist_name,artist_popularity,artist_followers,artist_genres,album_id,album_name,album_release_date,album_total_tracks,album_type,track_duration_min
0,3EJS5LyekDim1Tf5rBFmZl,Trippy Mane (ft. Project Pat),4,0,True,Diplo,77,2812821,moombahton,5QRFnGnBeMGePBKF2xTz5z,"d00mscrvll, Vol. 1",2025-10-31,9,album,1.55
1,1oQW6G2ZiwMuHqlPpP27DB,OMG!,1,0,True,Yelawolf,64,2363438,"country hip hop, southern hip hop",4SUmmwnv0xTjRcLdjczGg2,OMG!,2025-10-31,1,single,3.07
2,7mdkjzoIYlf1rx9EtBpGmU,Hard 2 Find,1,4,True,Riff Raff,48,193302,NaN,3E3zEAL8gUYWaLYB9L7gbp,Hard 2 Find,2025-10-31,1,single,2.55
3,67rW0Zl7oB3qEpD5YWWE5w,Still Get Like That (ft. Project Pat & Starrah),8,30,True,Diplo,77,2813710,moombahton,5QRFnGnBeMGePBKF2xTz5z,"d00mscrvll, Vol. 1",2025-10-31,9,album,1.69
4,15xptTfRBrjsppW0INUZjf,ride me like a harley,2,0,True,Rumelis,48,8682,dark r&b,06FDIpSHYmZAZoyuYtc7kd,come closer / ride me like a harley,2025-10-30,2,single,2.39


,valores_faltantes
track_id,0
track_name,0
track_popularity,0
artist_name,3
artist_popularity,0
artist_followers,0
artist_genres,3361
track_number,0
track_duration_min,0
album_total_tracks,0


## 3. Limpieza y preparación común

Se eliminan únicamente registros sin artista identificable o sin la variable objetivo (`track_popularity`). Los demás valores faltantes se conservan y se imputan dentro del pipeline del modelo cuando sea necesario.

Se crea `artist_key` para agrupar nombres ignorando diferencias de mayúsculas y espacios. Esto reduce duplicaciones por formato, aunque no resuelve homónimos: dos artistas distintos con el mismo nombre podrían quedar agrupados.

In [3]:
df = df_original.copy()

df["artist_name"] = df["artist_name"].astype("string").str.strip()
df = df.dropna(subset=["artist_name", "track_popularity"])
df = df[df["artist_name"] != ""]

columnas_numericas = [
    "track_popularity", "artist_popularity", "artist_followers",
    "track_number", "track_duration_min", "album_total_tracks"
]
for columna in columnas_numericas:
    df[columna] = pd.to_numeric(df[columna], errors="coerce")

df = df.dropna(subset=["track_popularity"])
df["artist_key"] = df["artist_name"].str.casefold().str.strip()

columnas_categoricas = ["artist_genres", "explicit", "album_type"]
for columna in columnas_categoricas:
    df[columna] = df[columna].astype("string").str.strip()
    df[columna] = df[columna].replace("", pd.NA)

df["album_release_date"] = pd.to_datetime(df["album_release_date"], errors="coerce")
df["release_year"] = df["album_release_date"].dt.year
df["release_month"] = df["album_release_date"].dt.month

print(f"Registros después de limpieza básica: {len(df):,}")
print(f"Nombres de artistas normalizados: {df['artist_key'].nunique():,}")
print(f"Track IDs duplicados: {df['track_id'].duplicated().sum():,}")

Registros después de limpieza básica: 8,579
Nombres de artistas normalizados: 2,547
Track IDs duplicados: 0


## 4. Perfiles agregados de artistas

Cada artista se resume mediante la mediana de popularidad y seguidores, el número de canciones distintas y la unión de los géneros disponibles. Se utiliza la mediana porque algunos atributos del artista presentan más de un valor entre las canciones asociadas al mismo nombre.

In [4]:
def reunir_generos(serie):
    # Une los géneros no nulos encontrados para un artista.
    generos = set()
    for valor in serie.dropna():
        for genero in str(valor).split(","):
            genero = genero.strip().casefold()
            if genero:
                generos.add(genero)
    return ", ".join(sorted(generos)) if generos else pd.NA


nombres_artistas = (
    df.groupby("artist_key")["artist_name"]
      .agg(lambda valores: valores.mode().iloc[0] if not valores.mode().empty else valores.iloc[0])
      .rename("artist_name")
)

perfiles = (
    df.groupby("artist_key")
      .agg(
          popularidad=("artist_popularity", "median"),
          seguidores=("artist_followers", "median"),
          canciones=("track_id", "nunique"),
          generos=("artist_genres", reunir_generos)
      )
      .join(nombres_artistas)
      .reset_index()
)

def clasificar_cobertura(numero_canciones):
    if numero_canciones <= 2:
        return "Información insuficiente"
    elif numero_canciones <= 4:
        return "Diagnóstico limitado"
    elif numero_canciones <= 9:
        return "Diagnóstico exploratorio completo"
    return "Mayor cobertura"

perfiles["categoria"] = perfiles["canciones"].apply(clasificar_cobertura)
artistas_validos = perfiles[perfiles["canciones"] >= UMBRAL_CANCIONES].copy()

print(f"Artistas con nombre registrado: {len(perfiles):,}")
print(f"Artistas con al menos {UMBRAL_CANCIONES} canciones: {len(artistas_validos):,}")
print(f"Canciones representadas por esos artistas: {artistas_validos['canciones'].sum():,}")

display(perfiles["categoria"].value_counts().rename_axis("categoria").to_frame("cantidad_artistas"))
display(artistas_validos.sort_values("canciones", ascending=False).head(10))

Artistas con nombre registrado: 2,547
Artistas con al menos 5 canciones: 336
Canciones representadas por esos artistas: 5,381


,cantidad_artistas
categoria,
Información insuficiente,1949
Diagnóstico limitado,262
Diagnóstico exploratorio completo,184
Mayor cobertura,152


,artist_key,popularidad,seguidores,canciones,generos,artist_name,categoria
2190,taylor swift,100.0,145443567.0,324,"country, folk, indie, pop",Taylor Swift,Mayor cobertura
2295,the weeknd,93.0,112957721.0,141,"alternative r&b, hip hop, r&b pop, synthpop",The Weeknd,Mayor cobertura
1265,lana del rey,88.0,51798111.0,99,"alternative pop, indie pop",Lana Del Rey,Mayor cobertura
144,ariana grande,90.0,107389514.0,94,pop,Ariana Grande,Mayor cobertura
1655,nirvana,81.0,23391272.0,91,"grunge, rock",Nirvana,Mayor cobertura
615,drake,95.0,103067383.0,84,rap,Drake,Mayor cobertura
1798,post malone,86.0,47720209.0,83,"country, hip hop, pop",Post Malone,Mayor cobertura
2253,the neighbourhood,83.0,21591569.0,77,<NA>,The Neighbourhood,Mayor cobertura
1258,lady gaga,88.0,42256604.0,70,"art pop, pop",Lady Gaga,Mayor cobertura
1700,olivia rodrigo,84.0,49050008.0,68,"alternative pop, pop punk, pop rock",Olivia Rodrigo,Mayor cobertura


### 4.1. Interpretación del umbral de cinco canciones

El umbral se aplica al diagnóstico competitivo, no al entrenamiento del modelo. El análisis previo encontró alrededor de 336 artistas con cinco o más canciones; la cifra exacta se vuelve a calcular aquí sobre el CSV cargado.

La decisión es práctica: con una o dos canciones hay poca información para describir el catálogo. Cinco canciones permiten un diagnóstico exploratorio más útil, pero no constituyen un estándar estadístico universal ni garantizan representatividad.

In [5]:
total_artistas = perfiles["artist_key"].nunique()
cantidad_validos = len(artistas_validos)
porcentaje_validos = 100 * cantidad_validos / total_artistas if total_artistas else 0

print(
    f"{cantidad_validos:,} de {total_artistas:,} artistas "
    f"({porcentaje_validos:.1f} %) tienen al menos {UMBRAL_CANCIONES} canciones."
)

336 de 2,547 artistas (13.2 %) tienen al menos 5 canciones.


## 5. Búsqueda de artistas comparables

La comparación considera popularidad, seguidores (transformados con logaritmo) y similitud de géneros mediante Jaccard cuando ambos artistas tienen géneros disponibles.

Los pesos (50 %, 30 % y 20 %) son una regla de diseño del prototipo, no pesos aprendidos por un modelo. Cuando no hay género disponible para uno de los artistas, esa dimensión se excluye y los pesos restantes se reescalan.

In [6]:
def convertir_a_conjunto_generos(valor):
    if pd.isna(valor) or not str(valor).strip():
        return set()
    return {g.strip().casefold() for g in str(valor).split(",") if g.strip()}


def similitud_jaccard(generos_a, generos_b):
    union = generos_a | generos_b
    if not union:
        return np.nan
    return len(generos_a & generos_b) / len(union)


def buscar_comparables(nombre_artista, cantidad=10):
    # Busca el artista dentro del grupo que cumple el umbral de canciones.
    objetivo_key = str(nombre_artista).strip().casefold()
    objetivo_df = artistas_validos[artistas_validos["artist_key"] == objetivo_key]

    if objetivo_df.empty:
        print(f"No se encontró '{nombre_artista}' entre los artistas con al menos {UMBRAL_CANCIONES} canciones.")
        return pd.DataFrame()

    objetivo = objetivo_df.iloc[0]
    candidatos = artistas_validos[artistas_validos["artist_key"] != objetivo_key].copy()

    if candidatos.empty:
        return candidatos

    pesos = {"popularidad": 0.50, "seguidores": 0.30, "generos": 0.20}
    max_seguidores = df["artist_followers"].max()
    max_log_seguidores = np.log1p(max(max_seguidores, 0)) if pd.notna(max_seguidores) else 0

    distancias = []
    for _, candidato in candidatos.iterrows():
        componentes = []
        pesos_activos = []

        if pd.notna(objetivo["popularidad"]) and pd.notna(candidato["popularidad"]):
            componentes.append(abs(candidato["popularidad"] - objetivo["popularidad"]) / 100)
            pesos_activos.append(pesos["popularidad"])

        if pd.notna(objetivo["seguidores"]) and pd.notna(candidato["seguidores"]) and max_log_seguidores > 0:
            log_objetivo = np.log1p(max(objetivo["seguidores"], 0))
            log_candidato = np.log1p(max(candidato["seguidores"], 0))
            componentes.append(abs(log_candidato - log_objetivo) / max_log_seguidores)
            pesos_activos.append(pesos["seguidores"])

        generos_objetivo = convertir_a_conjunto_generos(objetivo["generos"])
        generos_candidato = convertir_a_conjunto_generos(candidato["generos"])

        if generos_objetivo and generos_candidato:
            similitud = similitud_jaccard(generos_objetivo, generos_candidato)
            componentes.append(1 - similitud)
            pesos_activos.append(pesos["generos"])

        distancia = np.average(componentes, weights=pesos_activos) if pesos_activos else np.nan
        distancias.append(distancia)

    candidatos["distancia"] = distancias
    resultado = (
        candidatos.dropna(subset=["distancia"])
        .sort_values("distancia")
        .head(cantidad)
    )

    return resultado[
        ["artist_name", "popularidad", "seguidores", "canciones", "generos", "distancia"]
    ].reset_index(drop=True)


# Se utiliza un artista presente en la tabla como ejemplo reproducible.
ARTISTA_EJEMPLO = artistas_validos.iloc[0]["artist_name"] if not artistas_validos.empty else None
if ARTISTA_EJEMPLO is not None:
    print(f"Artistas comparables a: {ARTISTA_EJEMPLO}")
    display(buscar_comparables(ARTISTA_EJEMPLO, cantidad=10))

Artistas comparables a: 21 Savage


,artist_name,popularidad,seguidores,canciones,generos,distancia
0,Calvin Harris,84.0,23078931.0,21,edm,0.000620
1,Lil Baby,84.0,21644763.0,15,<NA>,0.001900
2,Arctic Monkeys,84.0,32059138.0,10,"garage rock, indie",0.005938
3,Cigarettes After Sex,84.0,17373704.0,9,dream pop,0.006285
4,Frank Ocean,83.5,20284390.5,6,<NA>,0.006320
5,Nicki Minaj,84.0,34294512.5,18,<NA>,0.007282
6,Twenty One Pilots,83.0,25761380.0,14,<NA>,0.007824
7,The Neighbourhood,83.0,21591569.0,77,<NA>,0.008199
8,Michael Jackson,84.0,37431031.0,21,<NA>,0.009028
9,YoungBoy Never Broke Again,84.0,14936207.0,5,<NA>,0.009301


## 6. Preparación del modelo de Machine Learning

La variable objetivo es `track_popularity`. No se incluyen identificadores ni nombres de canciones o artistas como predictores.

**Importante:** `artist_popularity` y `artist_followers` son variables de contexto relacionadas potencialmente con la popularidad de una canción. El modelo estima popularidad observada a partir de esos atributos; no pronostica popularidad futura.

In [7]:
target = "track_popularity"

numeric_features = [
    "artist_popularity", "artist_followers", "track_number",
    "track_duration_min", "album_total_tracks", "release_year", "release_month"
]
categorical_features = ["explicit", "artist_genres", "album_type"]
features = numeric_features + categorical_features

features_faltantes = [col for col in features if col not in df.columns]
if features_faltantes:
    raise ValueError(f"Faltan variables predictoras: {features_faltantes}")

X = df[features].copy()

# sklearn trabaja de forma más consistente con np.nan que con pd.NA
# en columnas categóricas de tipo pandas StringDtype.
for columna in categorical_features:
    X[columna] = X[columna].astype(object)
    X[columna] = X[columna].where(pd.notna(X[columna]), np.nan)

y = df[target].copy()
groups = df["artist_key"].copy()

print(f"Registros para modelar: {len(X):,}")
print(f"Artistas/grupos para la partición: {groups.nunique():,}")
print(f"Variables predictoras: {len(features)}")

Registros para modelar: 8,579
Artistas/grupos para la partición: 2,547
Variables predictoras: 10


## 7. Separación de entrenamiento y prueba por artista

`GroupShuffleSplit` evita que un mismo nombre normalizado de artista aparezca simultáneamente en entrenamiento y prueba. La evaluación mide el comportamiento con artistas no vistos durante el entrenamiento.

In [8]:
if groups.nunique() < 2:
    raise ValueError("Se necesitan al menos dos artistas para separar entrenamiento y prueba.")

splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()
groups_train, groups_test = groups.iloc[train_idx], groups.iloc[test_idx]

artistas_compartidos = set(groups_train) & set(groups_test)

print(f"Canciones para entrenamiento: {len(X_train):,}")
print(f"Canciones para prueba: {len(X_test):,}")
print(f"Artistas en entrenamiento: {groups_train.nunique():,}")
print(f"Artistas en prueba: {groups_test.nunique():,}")
print(f"Artistas compartidos entre conjuntos: {len(artistas_compartidos)}")

assert len(artistas_compartidos) == 0, "Hay artistas compartidos entre entrenamiento y prueba."

Canciones para entrenamiento: 6,935
Canciones para prueba: 1,644
Artistas en entrenamiento: 2,037
Artistas en prueba: 510
Artistas compartidos entre conjuntos: 0


## 8. Preprocesamiento

Los valores faltantes se imputan dentro del pipeline: mediana para variables numéricas y categoría más frecuente para variables categóricas. One-Hot Encoding convierte las categorías en columnas binarias e ignora categorías nuevas que aparezcan en prueba.

Al estar dentro del pipeline, los imputadores y el codificador se ajustan solamente con el conjunto de entrenamiento.

In [9]:
numeric_transformer = Pipeline(
    steps=[("imputer", SimpleImputer(strategy="median"))]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

## 9. Modelo base (Baseline)

El baseline predice la mediana de popularidad del conjunto de entrenamiento para todas las canciones de prueba. Sirve como referencia sencilla para evaluar si Random Forest aporta una mejora.

In [10]:
baseline = DummyRegressor(strategy="median")
baseline.fit(X_train, y_train)
baseline_predictions = baseline.predict(X_test)

## 10. Modelo Random Forest Regressor

Random Forest puede capturar relaciones no lineales entre variables numéricas y categóricas. Se mantienen hiperparámetros moderados para priorizar un entrenamiento reproducible; no se realiza una búsqueda extensa de hiperparámetros por el alcance introductorio del proyecto.

In [11]:
random_forest = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestRegressor(
            n_estimators=300,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            min_samples_leaf=2
        ))
    ]
)

random_forest.fit(X_train, y_train)
rf_predictions = random_forest.predict(X_test)

## 11. Evaluación de modelos

- **MAE:** error absoluto promedio, en puntos de popularidad.
- **RMSE:** penaliza más los errores grandes.
- **R²:** puede ser negativo si el modelo funciona peor que una referencia basada en la media.

Un MAE menor indica menor error promedio. El R² no debe utilizarse de forma aislada.

In [12]:
def evaluar_modelo(nombre, valores_reales, predicciones):
    return {
        "modelo": nombre,
        "MAE": mean_absolute_error(valores_reales, predicciones),
        "RMSE": np.sqrt(mean_squared_error(valores_reales, predicciones)),
        "R2": r2_score(valores_reales, predicciones)
    }


resultados_modelos = pd.DataFrame([
    evaluar_modelo("Baseline (mediana)", y_test, baseline_predictions),
    evaluar_modelo("Random Forest", y_test, rf_predictions)
])

display(resultados_modelos.round(3))

mae_baseline = resultados_modelos.loc[
    resultados_modelos["modelo"] == "Baseline (mediana)", "MAE"
].iloc[0]
mae_rf = resultados_modelos.loc[
    resultados_modelos["modelo"] == "Random Forest", "MAE"
].iloc[0]

if mae_rf < mae_baseline:
    print("Resultado: Random Forest mejora el MAE frente al baseline.")
else:
    print("Resultado: Random Forest NO mejora el MAE frente al baseline.")
    print("No se debe afirmar que el modelo supera al baseline.")

,modelo,MAE,RMSE,R2
0,Baseline (mediana),20.132,25.981,-0.091
1,Random Forest,15.370,20.904,0.294


Resultado: Random Forest mejora el MAE frente al baseline.


## 12. Predicciones y análisis de desviaciones

La desviación se define como `popularidad_real - popularidad_estimada`.

- Positiva: popularidad observada por encima de la estimada.
- Negativa: popularidad observada por debajo de la estimada.

Estas diferencias son señales para revisión, no evidencia de potencial futuro, causalidad o rentabilidad.

In [13]:
predicciones = pd.DataFrame({
    "track_id": df.iloc[test_idx]["track_id"].values,
    "cancion": df.iloc[test_idx]["track_name"].values,
    "artista": df.iloc[test_idx]["artist_name"].values,
    "popularidad_real": y_test.values,
    "popularidad_estimada": rf_predictions
})

predicciones["error_absoluto"] = (
    predicciones["popularidad_real"] - predicciones["popularidad_estimada"]
).abs()
predicciones["desviacion"] = (
    predicciones["popularidad_real"] - predicciones["popularidad_estimada"]
)

predicciones["tipo_desviacion"] = np.select(
    [predicciones["desviacion"] > 0, predicciones["desviacion"] < 0],
    ["Por encima de lo estimado", "Por debajo de lo estimado"],
    default="En línea con lo estimado"
)

display(predicciones.head(10))

,track_id,cancion,artista,popularidad_real,popularidad_estimada,error_absoluto,desviacion,tipo_desviacion
0,15xptTfRBrjsppW0INUZjf,ride me like a harley,Rumelis,0,36.430712,36.430712,-36.430712,Por debajo de lo estimado
1,1YEZbdT417SfolPQzaoHs2,come closer,Rumelis,27,39.101743,12.101743,-12.101743,Por debajo de lo estimado
2,4V1Sh9Y2z6e8HpzNOet5rC,PAPARAZZI ARRIVE - English Ver.,A2O,48,36.138026,11.861974,11.861974,Por encima de lo estimado
3,66F6YyfnWy6cNc6bd3zl2C,MELODY - CHENYU & SHIJIE & QUCHANG,A2O,27,33.778177,6.778177,-6.778177,Por debajo de lo estimado
4,22c6Yop5YVEAhw8UG6O1W1,My Muse,Leon Thomas,67,74.564882,7.564882,-7.564882,Por debajo de lo estimado
5,19A7srqQSXoW14muGyiHjg,BOSS - English Ver.,A2O,33,38.302074,5.302074,-5.302074,Por debajo de lo estimado
6,2zzjpwG8wu3skr08m7rM6e,Eggshell Tap Dancer,The All-American Rejects,52,48.385450,3.614550,3.614550,Por encima de lo estimado
7,2Zq3Trov36G0TtrUt4ZklL,superstar,Artemas,59,71.087263,12.087263,-12.087263,Por debajo de lo estimado
8,2H6cMXx0Kmj6XlbMx4BYi9,Trip,A2O,32,35.593392,3.593392,-3.593392,Por debajo de lo estimado
9,2zLnwikuCxJwMkEI8tqDN4,you never really cared,Anna Marx,31,38.136620,7.136620,-7.136620,Por debajo de lo estimado


### 12.1. Canciones con desviaciones positivas y negativas

Se muestran los casos más destacados del conjunto de prueba. Conviene explicar que el modelo no conoce campañas, inversión promocional, reproducciones históricas ni contexto externo que podría explicar esas diferencias.

In [14]:
canciones_por_encima = predicciones.sort_values("desviacion", ascending=False).head(10)
canciones_por_debajo = predicciones.sort_values("desviacion", ascending=True).head(10)

print("Canciones con popularidad observada más alta que la estimada:")
display(canciones_por_encima)

print("Canciones con popularidad observada más baja que la estimada:")
display(canciones_por_debajo)

Canciones con popularidad observada más alta que la estimada:


,track_id,cancion,artista,popularidad_real,popularidad_estimada,error_absoluto,desviacion,tipo_desviacion
665,6AGOKlMZWLCaEJGnaROtF9,Put Your Records On,Ritt Momney,67,25.850710,41.149290,41.149290,Por encima de lo estimado
1076,7H0ya83CMmgFcOhw0UB6ow,Space Song,Beach House,77,36.022075,40.977925,40.977925,Por encima de lo estimado
340,1XBYiRV30ykHw5f4wm6qEn,NIGHTS LIKE THIS,The Kid LAROI,86,48.769604,37.230396,37.230396,Por encima de lo estimado
1383,4356Typ82hUiFAynbLYbPn,DJ Got Us Fallin' In Love (feat. Pitbull),USHER,83,46.081649,36.918351,36.918351,Por encima de lo estimado
1482,7BHPGtpuuWWsvE7cCaMuEU,Naive,The Kooks,77,41.730803,35.269197,35.269197,Por encima de lo estimado
526,131SfRMv57LKf8xtaSt39k,Who Do You Want,Ex Habit,66,30.857770,35.142230,35.142230,Por encima de lo estimado
1288,5SXsXjVJCWeJuf7FHvgBYR,The Avengers,Alan Silvestri,61,26.221888,34.778112,34.778112,Por encima de lo estimado
1404,2QTDuJIGKUjR7E2Q6KupIh,Test Drive - From How To Train Your Dragon Music From The Motion Picture,John Powell,67,32.768643,34.231357,34.231357,Por encima de lo estimado
610,4l0RmWt52FxpVxMNni6i63,you broke me first,Tate McRae,85,50.841484,34.158516,34.158516,Por encima de lo estimado
1261,3cHyrEgdyYRjgJKSOiOtcS,Timber,Pitbull,82,48.615758,33.384242,33.384242,Por encima de lo estimado


Canciones con popularidad observada más baja que la estimada:


,track_id,cancion,artista,popularidad_real,popularidad_estimada,error_absoluto,desviacion,tipo_desviacion
1374,0PERS23jNsjcjBywXLvpyc,When I'm Small,Phantogram,0,69.044591,69.044591,-69.044591,Por debajo de lo estimado
1427,0zMAvsQmaQ4DGsmXtKixpn,Burning Pile,Mother Mother,1,69.930579,68.930579,-68.930579,Por debajo de lo estimado
497,6gI9b2VsoWhjhIuIeToDVs,Die Hard,Kendrick Lamar,2,70.543349,68.543349,-68.543349,Por debajo de lo estimado
1373,6TO82rF4nqsBTwhHoSvnue,Mouthful of Diamonds,Phantogram,0,68.528174,68.528174,-68.528174,Por debajo de lo estimado
653,4HffeEF97c6UxNePgbuECW,Pancakes for Dinner,Lizzy McAlpine,0,68.251877,68.251877,-68.251877,Por debajo de lo estimado
913,7i2DJ88J7jQ8K7zqFX2fW8,Now Or Never,Halsey,1,69.086724,68.086724,-68.086724,Por debajo de lo estimado
911,44n97yHySt0Z9rqPaXgjCK,Sorry,Halsey,1,68.549173,67.549173,-67.549173,Por debajo de lo estimado
1302,6UxSmGD3Ys7BIpbUpSERBM,Somebody That I Used To Know,Gotye,0,67.473123,67.473123,-67.473123,Por debajo de lo estimado
1065,5HuUOkUQEyMQMXnqqxOQJF,Same Old Love,Selena Gomez,0,67.157557,67.157557,-67.157557,Por debajo de lo estimado
559,5rW7qTn83sCKtqBoneJs63,edamame,bbno$,2,68.406742,66.406742,-66.406742,Por debajo de lo estimado


## 13. Limitaciones y controles de interpretación

1. **Cobertura del dataset:** no representa necesariamente todo Spotify ni todo el mercado musical.
2. **Popularidad de plataforma:** no equivale a ventas, ingresos, aceptación en vivo o éxito comercial.
3. **Géneros faltantes:** limitan las comparaciones por estilo.
4. **Identidad de artistas:** la agrupación se realiza por nombre normalizado, no por identificador único; puede haber homónimos.
5. **Variables de contexto:** popularidad y seguidores del artista influyen en la estimación; el modelo no mide exclusivamente atributos intrínsecos de la canción.
6. **Evaluación:** una sola partición por artista puede variar; reportar esta limitación.
7. **Desviaciones:** una canción por encima de lo estimado no necesariamente crecerá ni será rentable promocionarla.

### Dos limitaciones que se pueden discutir en la presentación
- El modelo puede equivocarse especialmente en canciones con características poco frecuentes o artistas con poca información.
- Los metadatos incompletos, especialmente los géneros, pueden reducir la calidad de las comparaciones entre artistas.

## 14. Exportación de resultados

In [15]:
perfiles.to_csv("perfiles_artistas.csv", index=False)
artistas_validos.to_csv("artistas_validos_5_o_mas_canciones.csv", index=False)
resultados_modelos.to_csv("comparacion_modelos.csv", index=False)
predicciones.to_csv("predicciones_random_forest_prueba.csv", index=False)

print("Archivos exportados:")
print("- perfiles_artistas.csv")
print("- artistas_validos_5_o_mas_canciones.csv")
print("- comparacion_modelos.csv")
print("- predicciones_random_forest_prueba.csv")

Archivos exportados:
- perfiles_artistas.csv
- artistas_validos_5_o_mas_canciones.csv
- comparacion_modelos.csv
- predicciones_random_forest_prueba.csv


## 15. Conclusión del análisis

Completar esta sección después de ejecutar el notebook. No afirmar que Random Forest es mejor si no obtiene un MAE inferior al baseline.

### Plantilla para redactar la conclusión
- El dataset analizado contiene ___ canciones y ___ artistas con nombre registrado.
- Se identificaron ___ artistas con al menos cinco canciones.
- El baseline obtuvo un MAE de ___ y Random Forest obtuvo un MAE de ___.
- El modelo [mejoró / no mejoró] el error frente al baseline.
- Las desviaciones positivas identifican canciones cuya popularidad observada supera la estimación del modelo; estos casos pueden priorizarse para revisión comercial.
- Los resultados son exploratorios y no constituyen predicciones de éxito futuro.